# 🔍 Deepfake Detection — Google Colab Training
**GPU runtime recommended** (Runtime → Change runtime type → T4 GPU)

This notebook:
1. Installs dependencies
2. Downloads the 140k Real & Fake Faces dataset
3. Trains EfficientNet-B4 for deepfake detection
4. Saves the model for use in the web app

In [ ]:
# ── 1. Install dependencies ──
!pip install -q timm albumentations kaggle

In [ ]:
# ── 2. Upload kaggle.json ──
# Go to kaggle.com → Account → Create New API Token → upload the file below
from google.colab import files
import os

uploaded = files.upload()   # upload kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
# ── 3. Download dataset ──
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p ./downloads --unzip
!ls ./downloads/real_vs_fake/real-vs-fake/

In [ ]:
# ── 4. Organise folders ──
import shutil
from pathlib import Path

src_root = Path('./downloads/real_vs_fake/real-vs-fake')

for split, dest_split in [('train', 'train'), ('valid', 'val'), ('test', 'val')]:
    for cls in ['real', 'fake']:
        src  = src_root / split / cls
        dest = Path('./data') / dest_split / cls
        if not src.exists(): continue
        dest.mkdir(parents=True, exist_ok=True)
        for f in src.glob('*'):
            shutil.copy2(f, dest / f.name)

for split in ['train', 'val']:
    for cls in ['real', 'fake']:
        p = Path('./data') / split / cls
        if p.exists(): print(f'{split}/{cls}: {len(list(p.iterdir()))} images')

In [ ]:
# ── 5. Clone / copy project ──
# If you uploaded the project as a zip:
# from google.colab import files; files.upload()
# !unzip deepfake-detection.zip

# Or paste src/train.py inline and run the cells below directly.
!ls

In [ ]:
# ── 6. Train (10 epochs on GPU ~25 min) ──
!python src/train.py \
    --data_dir  ./data \
    --model_dir ./models \
    --model_name efficientnet_b4 \
    --img_size  224 \
    --epochs    10 \
    --batch_size 32 \
    --lr 1e-4

In [ ]:
# ── 7. Download the trained model ──
from google.colab import files
files.download('./models/best_model.pth')

In [ ]:
# ── 8. (Optional) Quick test ──
import sys; sys.path.insert(0, '.')
from src.predict import predict
from PIL import Image
import torch

img = Image.open('./data/val/fake/00001.jpg')  # swap with any image
result = predict(img, './models/best_model.pth', torch.device('cuda'))
print(result)